We will work with the Samsung Human Activity Recognition dataset. The data comes from accelerometers and gyros of Samsung Galaxy S3 mobile phones (you can find more info about the features using the link above), the type of activity of a person with a phone in his/her pocket is also known – whether he/she walked, stood, lay, sat or walked up or down the stairs.

In [1]:
import os 
from zipfile import ZipFile
from pathlib import Path
import requests

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm_notebook

%matplotlib inline
from matplotlib import pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (12, 9)
plt.rcParams["font.family"] = "DejaVu Sans"

from sklearn import metrics
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 17

In [2]:
def load_dataset(extract_path:Path, filename:str, overwrite=False):
    filepath = extract_path/filename
    if filepath.exists() and not overwrite:
        print("dataset already in set")
        return

    with ZipFile(filepath, "r") as zipObj:
        zipObj.extractall(extract_path)

In [3]:
FILE_NAME = "UCI HAR Dataset"
DATA_PATH = Path("./")
load_dataset(extract_path=DATA_PATH, filename=FILE_NAME)

PATH_TO_SAMSUNG_DATA = DATA_PATH / FILE_NAME.strip(".zip")
print(PATH_TO_SAMSUNG_DATA)

dataset already in set
UCI HAR Dataset


In [4]:
X_train = np.loadtxt(PATH_TO_SAMSUNG_DATA / "train" / "X_train.txt")
y_train = np.loadtxt(PATH_TO_SAMSUNG_DATA / "train" / "y_train.txt").astype(int)

X_test = np.loadtxt(PATH_TO_SAMSUNG_DATA / "test" / "X_test.txt")
y_test = np.loadtxt(PATH_TO_SAMSUNG_DATA / "test" / "y_test.txt").astype(int)

In [5]:
X_train.shape, y_train.shape

((7352, 561), (7352,))

In [6]:
# Checking dimensions
assert X_train.shape == (7352, 561) and y_train.shape == (7352,)
assert X_test.shape == (2947, 561) and y_test.shape == (2947,)

For clustering, we do not need a target vector, so we’ll work with the combination of training and test samples. Merge X_train with X_test, and y_train with y_test.

In [ ]:
X = np.vstack([X_train, X_test])
y = np.hstack([y_train, y_test])
# y[:5]


array([5, 5, 5, 5, 5])

Define the number of unique values of the labels of the target class.

In [15]:
np.unique(y)

array([1, 2, 3, 4, 5, 6])

These labels correspond to:

1 – walking

2 – walking upstairs

3 – walking downstairs

4 – sitting

5 – standing

6 – laying down

In [16]:
n_classes = np.unique(y).size

Scale the sample using StandardScaler with default parameters.

In [17]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Reduce the number of dimensions using PCA, leaving as many components as necessary to explain at least 90% of the variance of the original (scaled) data. Use the scaled dataset and fix random_state (RANDOM_STATE constant).